# Do ROUGE-L and BERTScore notice a wrong-paper abstract?

A check of the evaluation measures (Section 6.2 of the thesis). For 200 evaluation papers, an
untrained language model wrote three abstracts from an identical prompt that lists either the
paper's real references, the same number of random papers, or random papers from the paper's
own arXiv category. All abstracts are scored against the paper's original abstract.

Input: `experiments/2026-11-16_metric-check-random-citations/`.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import thesis_style as ts
ts.apply()

E = "../experiments/2026-11-16_metric-check-random-citations/"
CONDITIONS = {"real": "real citations", "same_category": "random citations (same category)",
              "random": "random citations (any field)"}
scored = {c: pd.read_csv(E + c + "/scored.csv").set_index("node_id") for c in CONDITIONS}
common = sorted(set.intersection(*(set(df.index) for df in scored.values())))
scored = {c: df.loc[common] for c, df in scored.items()}
print(f"{len(common)} papers with all three abstracts scored")

200 papers with all three abstracts scored


## Scores per condition

In [2]:
METRICS = {"rougeL_f": "ROUGE-L F1", "bertscore_f1": "BERTScore F1",
           "content": "judge content agreement (0-3)", "tfidf_top1": "TF-IDF top-1 (of 200)"}

def auc(a, b):
    a, b = np.asarray(a)[:, None], np.asarray(b)[None, :]
    return (a > b).mean() + 0.5 * (a == b).mean()

real = scored["real"]
rows = []
for control in ("same_category", "random"):
    other = scored[control]
    for col, name in METRICS.items():
        a, b = pd.to_numeric(real[col], errors="coerce"), pd.to_numeric(other[col], errors="coerce")
        ok = a.notna() & b.notna()
        a, b = a[ok], b[ok]
        d = a - b
        rows.append({"control": control, "metric": name, "papers": len(d), "real": a.mean(),
                     "control mean": b.mean(), "difference": d.mean(),
                     "SE": d.std(ddof=1) / np.sqrt(len(d)),
                     "control as % of real": 100 * b.mean() / a.mean(),
                     "real > control (%)": 100 * (d > 0).mean(), "equal (%)": 100 * (d == 0).mean(),
                     "AUC": auc(a, b)})
table = pd.DataFrame(rows).set_index(["control", "metric"])
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 220):
    print(table.to_string())

print()
for c, df in scored.items():
    w = df["generated_abstract"].astype(str).str.split().str.len()
    parts = ", ".join(f"{k} {pd.to_numeric(df[k]).mean():.2f}" for k in ("subfield", "problem", "task"))
    print(f"{c}: judge {parts}; content score 0 for {100 * (df['content'] == 0).mean():.1f}% of papers; "
          f"generated words median {w.median():.0f}")
print(f"original abstracts: words median {real['original_abstract'].astype(str).str.split().str.len().median():.0f}")

                                             papers  real  control mean  difference    SE  control as % of real  real > control (%)  equal (%)   AUC
control       metric                                                                                                                                
same_category ROUGE-L F1                        200 0.153         0.112       0.041 0.002                73.302              94.500      1.000 0.932
              BERTScore F1                      200 0.841         0.822       0.019 0.001                97.769              97.000      0.000 0.920
              judge content agreement (0-3)     200 2.030         0.252       1.778 0.061                12.397              94.500      4.500 0.959
              TF-IDF top-1 (of 200)             200 0.760         0.025       0.735 0.031                 3.289              73.500     26.500 0.867
random        ROUGE-L F1                        200 0.153         0.112       0.041 0.002                7

## Constructed sentence pairs

Short statements written for illustration, scored by `example_pairs.py` in the experiment folder.

In [3]:
pairs = pd.read_csv(E + "example_pairs.csv")
with pd.option_context("display.max_colwidth", 120, "display.width", 250):
    print(pairs.round(3).to_string(index=False))

                         kind                                                                                                  text_a                                                                                                 text_b  rougeL_f  bertscore_f1
             reversed finding Adding residual connections improves the accuracy of deep graph neural networks on node classification. Adding residual connections reduces the accuracy of deep graph neural networks on node classification.     0.929         0.994
                swapped roles                     We use a large language model to generate training data for a graph neural network.                    We use a graph neural network to generate training data for a large language model.     0.600         0.974
        different application                                 We propose a transformer model for detecting tumours in medical images.                          We propose a transformer model for detecting fraud in fina